In [ ]:
# NovaBridge AI Project Management System

**Portfolio Project — AI-Powered Project Risk Analysis, PM Copilot & Workflow Automation**

NovaBridge analyzes project data from Excel, calculates deterministic project and dependency risks, uses AI to interpret those risks, validates AI outputs, provides a conversational PM Copilot, and supports human-approved workflow escalation through n8n and Gmail.

## System Architecture

Excel Project Data
→ Data Validation
→ Python Deterministic Risk Engine
→ Structured AI Risk Analysis
→ Grounding & Quality Control
→ AI PM Copilot
→ Human Approval
→ n8n Workflow Automation
→ Gmail Escalation

## Core Design Principle

**Python calculates facts. AI interprets facts.**

Deterministic Python is responsible for deadlines, overdue calculations, completion rates, dependencies, project health, and risk scores.

AI is responsible for interpretation, prioritization, recommendations, executive summaries, and conversational access.

## 🔐 Public Portfolio Safety Notice

This GitHub version intentionally contains **no API keys or webhook URLs**.

- `OPENAI_API_KEY` should be stored in **Google Colab Secrets**.
- `N8N_PRODUCTION_WEBHOOK_URL` should be stored in **Google Colab Secrets**.
- Live escalation execution is disabled by default to prevent accidental emails or workflow actions.
- The project uses synthetic project-management data for demonstration purposes.

> **Python calculates facts. AI interprets facts. Human approval controls external actions.**


In [ ]:
# 1. Environment & Configuration

This section initializes the Python environment, imports required libraries, configures the OpenAI client, and loads secure configuration required by NovaBridge.

In [ ]:
# ============================================================
# NOVABRIDGE AI PROJECT MANAGEMENT SYSTEM
# PRODUCTION PIPELINE
# ============================================================

# ------------------------------------------------------------
# CELL 1 - SYSTEM SETUP
# ------------------------------------------------------------

!pip install -q openai openpyxl


# Standard Python libraries
import os
import json

from datetime import datetime
from getpass import getpass


# Data processing
import pandas as pd


# OpenAI
from openai import OpenAI


print("==========================================")
print("NOVABRIDGE AI PROJECT MANAGEMENT SYSTEM")
print("==========================================")

print("\nProduction Pipeline")

print("\n✅ Required libraries loaded successfully.")

In [ ]:
# ============================================================
# CELL 2 - NOVABRIDGE SYSTEM CONFIGURATION
# ============================================================


# ------------------------------------------------------------
# 1. SYSTEM INFORMATION
# ------------------------------------------------------------

SYSTEM_NAME = "NovaBridge AI Project Management System"

COMPANY_NAME = "NovaBridge Technologies"

SYSTEM_VERSION = "1.0"


# ------------------------------------------------------------
# 2. AI MODEL CONFIGURATION
# ------------------------------------------------------------

AI_MODEL = "gpt-5.6-luna"


# ------------------------------------------------------------
# 3. EXCEL CONFIGURATION
# ------------------------------------------------------------

EXCEL_SHEET_NAME = "Tasks"


REQUIRED_COLUMNS = [

    "task_id",

    "project_id",

    "project_name",

    "task_name",

    "owner",

    "deadline",

    "status",

    "priority",

    "dependency",

    "latest_update"
]


# ------------------------------------------------------------
# 4. ALLOWED VALUES
# ------------------------------------------------------------

ALLOWED_STATUSES = [

    "Complete",

    "In Progress",

    "Delayed",

    "Not Started"
]


ALLOWED_PRIORITIES = [

    "Critical",

    "High",

    "Medium",

    "Low"
]


# ------------------------------------------------------------
# 5. ANALYSIS DATE CONFIGURATION
# ------------------------------------------------------------

# None means:
# NovaBridge automatically uses today's date.
#
# Later, if we want to simulate a historical date,
# we can enter something like:
#
# ANALYSIS_DATE_OVERRIDE = "2026-09-15"

ANALYSIS_DATE_OVERRIDE = None


# ------------------------------------------------------------
# 6. DISPLAY CONFIGURATION
# ------------------------------------------------------------

SHOW_DETAILED_VALIDATION = True


# ------------------------------------------------------------
# 7. VERIFY CONFIGURATION
# ------------------------------------------------------------

print("==========================================")
print("NOVABRIDGE CONFIGURATION")
print("==========================================")

print("\nSystem:")
print(SYSTEM_NAME)

print("\nVersion:")
print(SYSTEM_VERSION)

print("\nCompany:")
print(COMPANY_NAME)

print("\nAI Model:")
print(AI_MODEL)

print("\nExcel Sheet:")
print(EXCEL_SHEET_NAME)

print("\nRequired Excel Columns:")
print(len(REQUIRED_COLUMNS))

print("\nAnalysis Date Override:")
print(ANALYSIS_DATE_OVERRIDE)

print("\n✅ NovaBridge configuration loaded.")

In [ ]:
# ============================================================
# CELL 3 - SECURE OPENAI CONNECTION
# ============================================================


# ------------------------------------------------------------
# 1. CHECK FOR EXISTING API KEY
# ------------------------------------------------------------

from google.colab import userdata

# Securely load OpenAI API key from Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get(
    "OPENAI_API_KEY"
)

if os.getenv("OPENAI_API_KEY"):
    print("✅ OpenAI API key loaded securely.")
else:
    raise ValueError(
        "OPENAI_API_KEY is not configured."
    )


# ------------------------------------------------------------
# 2. CREATE OPENAI CLIENT
# ------------------------------------------------------------

client = OpenAI()


# ------------------------------------------------------------
# 3. TEST CONNECTION
# ------------------------------------------------------------

print(
    "\nTesting OpenAI connection..."
)


try:

    test_response = client.responses.create(

        model=AI_MODEL,

        input=(
            "Reply with exactly: "
            "NovaBridge production connection successful."
        )
    )


    print(
        "\n",
        test_response.output_text
    )


    print(
        "\n✅ OpenAI connection ready."
    )


except Exception as error:

    print(
        "\n❌ OpenAI connection failed."
    )

    print(
        "\nError:"
    )

    print(error)

In [ ]:
# ============================================================
# CELL 4 - EXCEL INPUT & VALIDATION ENGINE
# ============================================================


def load_and_validate_excel(excel_file):

    print("==========================================")
    print("NOVABRIDGE EXCEL VALIDATION")
    print("==========================================")

    print("\nFile:")
    print(excel_file)


    # --------------------------------------------------------
    # 1. LOAD EXCEL
    # --------------------------------------------------------

    try:

        df = pd.read_excel(
            excel_file,
            sheet_name=EXCEL_SHEET_NAME
        )

    except Exception as error:

        raise ValueError(
            f"Unable to read Excel file: {error}"
        )


    print("\n✅ Excel file loaded.")


    # --------------------------------------------------------
    # 2. CHECK REQUIRED COLUMNS
    # --------------------------------------------------------

    missing_columns = [

        column

        for column in REQUIRED_COLUMNS

        if column not in df.columns
    ]


    if missing_columns:

        raise ValueError(
            "Missing required columns: "
            + ", ".join(missing_columns)
        )


    print("✅ Required columns validated.")


    # --------------------------------------------------------
    # 3. CHECK FOR EMPTY TASK IDS
    # --------------------------------------------------------

    if df["task_id"].isna().any():

        raise ValueError(
            "One or more task_id values are blank."
        )


    # --------------------------------------------------------
    # 4. CHECK FOR DUPLICATE TASK IDS
    # --------------------------------------------------------

    duplicate_task_ids = (
        df[
            df["task_id"].duplicated(
                keep=False
            )
        ]["task_id"]
        .astype(str)
        .tolist()
    )


    if duplicate_task_ids:

        raise ValueError(
            "Duplicate task IDs found: "
            + ", ".join(
                sorted(
                    set(
                        duplicate_task_ids
                    )
                )
            )
        )


    print("✅ Task IDs validated.")


    # --------------------------------------------------------
    # 5. VALIDATE STATUS VALUES
    # --------------------------------------------------------

    invalid_statuses = sorted(

        set(
            df[
                ~df["status"].isin(
                    ALLOWED_STATUSES
                )
            ]["status"]
            .dropna()
            .astype(str)
            .tolist()
        )
    )


    if invalid_statuses:

        raise ValueError(
            "Invalid status values found: "
            + ", ".join(invalid_statuses)
        )


    print("✅ Status values validated.")


    # --------------------------------------------------------
    # 6. VALIDATE PRIORITY VALUES
    # --------------------------------------------------------

    invalid_priorities = sorted(

        set(
            df[
                ~df["priority"].isin(
                    ALLOWED_PRIORITIES
                )
            ]["priority"]
            .dropna()
            .astype(str)
            .tolist()
        )
    )


    if invalid_priorities:

        raise ValueError(
            "Invalid priority values found: "
            + ", ".join(
                invalid_priorities
            )
        )


    print("✅ Priority values validated.")


    # --------------------------------------------------------
    # 7. VALIDATE DEADLINES
    # --------------------------------------------------------

    converted_deadlines = pd.to_datetime(
        df["deadline"],
        errors="coerce"
    )


    invalid_deadline_rows = (
        converted_deadlines.isna()
        & df["deadline"].notna()
    )


    if invalid_deadline_rows.any():

        invalid_rows = (
            df[
                invalid_deadline_rows
            ]["task_id"]
            .astype(str)
            .tolist()
        )

        raise ValueError(
            "Invalid deadlines found for tasks: "
            + ", ".join(invalid_rows)
        )


    if converted_deadlines.isna().any():

        raise ValueError(
            "One or more deadlines are blank."
        )


    df["deadline"] = (
        converted_deadlines
    )


    print("✅ Deadlines validated.")


    # --------------------------------------------------------
    # 8. NORMALIZE TASK IDs & DEPENDENCIES
    # --------------------------------------------------------

    df["task_id"] = (
        df["task_id"]
        .astype(str)
        .str.strip()
    )


    df["project_id"] = (
        df["project_id"]
        .astype(str)
        .str.strip()
    )


    df["dependency"] = (
        df["dependency"]
        .apply(
            lambda value:
            None
            if pd.isna(value)
            else str(value).strip()
        )
    )


    # --------------------------------------------------------
    # 9. VALIDATE DEPENDENCY REFERENCES
    # --------------------------------------------------------

    task_ids = set(
        df["task_id"]
    )


    invalid_dependencies = []


    for _, row in df.iterrows():

        dependency = row["dependency"]

        if (
            dependency is not None
            and dependency not in task_ids
        ):

            invalid_dependencies.append(
                (
                    row["task_id"],
                    dependency
                )
            )


    if invalid_dependencies:

        error_messages = [

            f"{task_id} → {dependency}"

            for task_id, dependency
            in invalid_dependencies
        ]

        raise ValueError(
            "Invalid dependency references: "
            + ", ".join(
                error_messages
            )
        )


    print("✅ Dependency references validated.")


    # --------------------------------------------------------
    # 10. FINAL RESULT
    # --------------------------------------------------------

    print("\n------------------------------------------")
    print("VALIDATION RESULT")
    print("------------------------------------------")

    print("\nTasks detected:")
    print(len(df))

    print("\nProjects detected:")
    print(
        df["project_id"].nunique()
    )

    print(
        "\n✅ Excel validation PASSED."
    )


    return df

In [ ]:
# ============================================================
# CELL 5 - BUILD NOVABRIDGE DATA STRUCTURES
# ============================================================


def build_novabridge_data(df):

    # --------------------------------------------------------
    # 1. BUILD TASK LIST
    # --------------------------------------------------------

    tasks = []


    for _, row in df.iterrows():

        task = {

            "task_id":
                str(row["task_id"]).strip(),

            "project_id":
                str(row["project_id"]).strip(),

            "project_name":
                str(row["project_name"]).strip(),

            "task_name":
                str(row["task_name"]).strip(),

            "owner":
                str(row["owner"]).strip(),

            "deadline":
                row["deadline"].strftime(
                    "%Y-%m-%d"
                ),

            "status":
                str(row["status"]).strip(),

            "priority":
                str(row["priority"]).strip(),

            "dependency":
                row["dependency"],

            "latest_update":
                str(row["latest_update"]).strip()
        }


        tasks.append(task)


    # --------------------------------------------------------
    # 2. BUILD PROJECT LIST
    # --------------------------------------------------------

    projects = []

    seen_projects = set()


    for task in tasks:

        project_id = task[
            "project_id"
        ]


        if (
            project_id
            not in seen_projects
        ):

            projects.append({

                "project_id":
                    project_id,

                "project_name":
                    task["project_name"]
            })


            seen_projects.add(
                project_id
            )


    # --------------------------------------------------------
    # 3. BUILD TASK LOOKUP
    # --------------------------------------------------------

    task_lookup = {

        task["task_id"]: task

        for task in tasks
    }


    # --------------------------------------------------------
    # 4. BUILD PROJECT LOOKUP
    # --------------------------------------------------------

    project_lookup = {

        project["project_id"]:
            project

        for project in projects
    }


    # --------------------------------------------------------
    # 5. BUILD REVERSE DEPENDENCY MAP
    # --------------------------------------------------------

    reverse_dependencies = {}


    for task in tasks:

        dependency = task[
            "dependency"
        ]


        if dependency:

            if (
                dependency
                not in reverse_dependencies
            ):

                reverse_dependencies[
                    dependency
                ] = []


            reverse_dependencies[
                dependency
            ].append(
                task["task_id"]
            )


    # --------------------------------------------------------
    # 6. RETURN EVERYTHING
    # --------------------------------------------------------

    return {

        "tasks":
            tasks,

        "projects":
            projects,

        "task_lookup":
            task_lookup,

        "project_lookup":
            project_lookup,

        "reverse_dependencies":
            reverse_dependencies
    }

In [ ]:
# ============================================================
# CELL 6 - DETERMINISTIC PROJECT RISK ENGINE
# ============================================================


def run_deterministic_engine(
    data,
    analysis_date
):

    tasks = data["tasks"]
    projects = data["projects"]
    task_lookup = data["task_lookup"]
    reverse_dependencies = data[
        "reverse_dependencies"
    ]


    # --------------------------------------------------------
    # 1. FIND DOWNSTREAM TASKS
    # --------------------------------------------------------

    def find_downstream_tasks(
        task_id,
        visited=None
    ):

        if visited is None:
            visited = set()

        downstream = []

        for child_id in reverse_dependencies.get(
            task_id,
            []
        ):

            if child_id not in visited:

                visited.add(child_id)

                downstream.append(
                    child_id
                )

                downstream.extend(
                    find_downstream_tasks(
                        child_id,
                        visited
                    )
                )

        return downstream


    # --------------------------------------------------------
    # 2. CHECK WHETHER TASK IS OVERDUE
    # --------------------------------------------------------

    def is_task_overdue(task):

        deadline = datetime.strptime(
            task["deadline"],
            "%Y-%m-%d"
        )

        return (
            deadline < analysis_date
            and task["status"] != "Complete"
        )


    # --------------------------------------------------------
    # 3. CALCULATE DAYS OVERDUE
    # --------------------------------------------------------

    def calculate_days_overdue(task):

        deadline = datetime.strptime(
            task["deadline"],
            "%Y-%m-%d"
        )

        if is_task_overdue(task):

            return (
                analysis_date - deadline
            ).days

        return 0


    # --------------------------------------------------------
    # 4. BLOCKER SEVERITY SCORE
    # --------------------------------------------------------

    def calculate_blocker_severity(
        task
    ):

        score = 0

        priority = task["priority"]

        days_overdue = (
            calculate_days_overdue(
                task
            )
        )


        if priority == "Critical":
            score += 3

        elif priority == "High":
            score += 2

        else:
            score += 1


        if days_overdue >= 7:
            score += 3

        elif days_overdue >= 3:
            score += 2

        elif days_overdue > 0:
            score += 1


        if task["status"] == "Delayed":
            score += 1


        if score >= 6:
            severity = "CRITICAL"

        elif score >= 4:
            severity = "HIGH"

        elif score >= 2:
            severity = "MEDIUM"

        else:
            severity = "LOW"


        return score, severity


    # --------------------------------------------------------
    # 5. DEPENDENCY CHAIN IMPACT
    # --------------------------------------------------------

    def calculate_chain_impact(
        task
    ):

        downstream_ids = (
            find_downstream_tasks(
                task["task_id"]
            )
        )


        critical_downstream = 0


        for downstream_id in downstream_ids:

            downstream_task = (
                task_lookup[
                    downstream_id
                ]
            )

            if (
                downstream_task[
                    "priority"
                ]
                == "Critical"
            ):

                critical_downstream += 1


        score = 0


        if task["priority"] == "Critical":
            score += 3

        elif task["priority"] == "High":
            score += 2

        else:
            score += 1


        score += len(
            downstream_ids
        )


        score += (
            critical_downstream * 2
        )


        if score >= 10:
            impact = "CRITICAL"

        elif score >= 7:
            impact = "HIGH"

        elif score >= 4:
            impact = "MEDIUM"

        else:
            impact = "LOW"


        return {

            "score":
                score,

            "impact":
                impact,

            "downstream_tasks":
                downstream_ids,

            "critical_downstream_tasks":
                critical_downstream
        }


    # --------------------------------------------------------
    # 6. PROJECT HEALTH
    # --------------------------------------------------------

    project_summaries = []


    for project in projects:

        project_id = (
            project["project_id"]
        )


        project_tasks = [

            task

            for task in tasks

            if task["project_id"]
            == project_id
        ]


        total_tasks = len(
            project_tasks
        )


        completed_tasks = sum(

            1

            for task in project_tasks

            if task["status"]
            == "Complete"
        )


        delayed_tasks = sum(

            1

            for task in project_tasks

            if task["status"]
            == "Delayed"
        )


        overdue_tasks = sum(

            1

            for task in project_tasks

            if is_task_overdue(
                task
            )
        )


        completion_percent = (

            completed_tasks
            / total_tasks
            * 100

            if total_tasks > 0

            else 0
        )


        if (
            delayed_tasks >= 2
            or overdue_tasks >= 2
        ):

            project_health = (
                "RED - Critical"
            )


        elif (
            delayed_tasks == 1
            or overdue_tasks == 1
        ):

            project_health = (
                "YELLOW - At Risk"
            )


        else:

            project_health = (
                "GREEN - On Track"
            )


        project_summaries.append({

            "project_id":
                project_id,

            "project_name":
                project[
                    "project_name"
                ],

            "total_tasks":
                total_tasks,

            "completed_tasks":
                completed_tasks,

            "delayed_tasks":
                delayed_tasks,

            "overdue_tasks":
                overdue_tasks,

            "completion_percent":
                round(
                    completion_percent,
                    1
                ),

            "project_health":
                project_health
        })


    # --------------------------------------------------------
    # 7. DIRECT DEPENDENCY RISKS
    # --------------------------------------------------------

    dependency_risks = []


    for task in tasks:

        if (
            task["status"] == "Delayed"
            or is_task_overdue(task)
        ):

            directly_blocked = (
                reverse_dependencies.get(
                    task["task_id"],
                    []
                )
            )


            if directly_blocked:

                severity_score, severity = (
                    calculate_blocker_severity(
                        task
                    )
                )


                dependency_risks.append({

                    "project_id":
                        task[
                            "project_id"
                        ],

                    "project_name":
                        task[
                            "project_name"
                        ],

                    "task_id":
                        task[
                            "task_id"
                        ],

                    "task_name":
                        task[
                            "task_name"
                        ],

                    "owner":
                        task[
                            "owner"
                        ],

                    "status":
                        task[
                            "status"
                        ],

                    "priority":
                        task[
                            "priority"
                        ],

                    "deadline":
                        task[
                            "deadline"
                        ],

                    "days_overdue":
                        calculate_days_overdue(
                            task
                        ),

                    "severity_score":
                        severity_score,

                    "severity":
                        severity,

                    "directly_blocked_tasks":
                        directly_blocked,

                    "latest_update":
                        task[
                            "latest_update"
                        ]
                })


    # --------------------------------------------------------
    # 8. DEPENDENCY CHAIN RISKS
    # --------------------------------------------------------

    dependency_chain_risks = []


    for task in tasks:

        if (
            task["status"] == "Delayed"
            or is_task_overdue(task)
        ):

            chain = (
                calculate_chain_impact(
                    task
                )
            )


            if chain[
                "downstream_tasks"
            ]:

                dependency_chain_risks.append({

                    "project_id":
                        task[
                            "project_id"
                        ],

                    "project_name":
                        task[
                            "project_name"
                        ],

                    "root_task_id":
                        task[
                            "task_id"
                        ],

                    "root_task_name":
                        task[
                            "task_name"
                        ],

                    "owner":
                        task[
                            "owner"
                        ],

                    "days_overdue":
                        calculate_days_overdue(
                            task
                        ),

                    "chain_impact_score":
                        chain[
                            "score"
                        ],

                    "chain_impact":
                        chain[
                            "impact"
                        ],

                    "downstream_tasks":
                        chain[
                            "downstream_tasks"
                        ],

                    "critical_downstream_tasks":
                        chain[
                            "critical_downstream_tasks"
                        ],

                    "latest_update":
                        task[
                            "latest_update"
                        ]
                })


    # --------------------------------------------------------
    # 9. RETURN ENGINE RESULTS
    # --------------------------------------------------------

    return {

        "project_summaries":
            project_summaries,

        "dependency_risks":
            dependency_risks,

        "dependency_chain_risks":
            dependency_chain_risks
    }

In [ ]:
# ============================================================
# CELL 7 - AI INPUT + STRUCTURED RISK ANALYST
# ============================================================


def run_ai_risk_analyst(
    data,
    engine_results,
    analysis_date
):

    # --------------------------------------------------------
    # 1. BUILD AI INPUT PACKAGE
    # --------------------------------------------------------

    ai_input_data = {

        "analysis_metadata": {
            "system":
                SYSTEM_NAME,

            "company":
                COMPANY_NAME,

            "analysis_date":
                analysis_date.strftime(
                    "%Y-%m-%d"
                ),

            "data_source":
                "Validated Excel Project Data",

            "system_version":
                SYSTEM_VERSION
        },

        "portfolio_summary": {

            "number_of_projects":
                len(
                    data["projects"]
                ),

            "number_of_tasks":
                len(
                    data["tasks"]
                ),

            "number_of_dependency_risks":
                len(
                    engine_results[
                        "dependency_risks"
                    ]
                ),

            "number_of_dependency_chain_risks":
                len(
                    engine_results[
                        "dependency_chain_risks"
                    ]
                )
        },

        "project_health":
            engine_results[
                "project_summaries"
            ],

        "direct_dependency_risks":
            engine_results[
                "dependency_risks"
            ],

        "dependency_chain_risks":
            engine_results[
                "dependency_chain_risks"
            ]
    }


    ai_input_json = json.dumps(
        ai_input_data,
        indent=2
    )


    # --------------------------------------------------------
    # 2. DEFINE STRUCTURED OUTPUT SCHEMA
    # --------------------------------------------------------

    risk_schema = {

        "type": "object",

        "properties": {

            "portfolio_health": {
                "type": "string",
                "enum": [
                    "GREEN",
                    "YELLOW",
                    "RED"
                ]
            },

            "executive_summary": {
                "type": "string"
            },

            "top_risks": {

                "type": "array",

                "items": {

                    "type": "object",

                    "properties": {

                        "project_id": {
                            "type": "string"
                        },

                        "project_name": {
                            "type": "string"
                        },

                        "root_task_id": {
                            "type": "string"
                        },

                        "risk_title": {
                            "type": "string"
                        },

                        "business_impact": {
                            "type": "string"
                        },

                        "management_attention": {

                            "type": "string",

                            "enum": [
                                "IMMEDIATE",
                                "HIGH",
                                "MEDIUM",
                                "LOW"
                            ]
                        },

                        "recommended_actions": {

                            "type": "array",

                            "items": {
                                "type": "string"
                            }
                        }
                    },

                    "required": [
                        "project_id",
                        "project_name",
                        "root_task_id",
                        "risk_title",
                        "business_impact",
                        "management_attention",
                        "recommended_actions"
                    ],

                    "additionalProperties":
                        False
                }
            },

            "information_gaps": {

                "type": "array",

                "items": {
                    "type": "string"
                }
            }
        },

        "required": [
            "portfolio_health",
            "executive_summary",
            "top_risks",
            "information_gaps"
        ],

        "additionalProperties":
            False
    }


    # --------------------------------------------------------
    # 3. AI INSTRUCTIONS
    # --------------------------------------------------------

    instructions = """
You are the AI Project Risk Analyst for
NovaBridge Technologies.

You receive structured project-management analysis
from a deterministic Python analytics engine.

IMPORTANT RULES:

1. Use only facts contained in the supplied data.

2. Python is the source of truth for:
   - deadlines
   - days overdue
   - project health
   - completion percentages
   - dependency relationships
   - severity scores
   - dependency-chain impact scores

3. Do not invent:
   - task IDs
   - project IDs
   - owners
   - dates
   - completion percentages
   - dependencies
   - scores

4. A task can be Delayed even when it has
   zero days overdue.

5. Never describe a task as overdue if
   days_overdue is 0.

6. Prioritize risks that affect important
   downstream dependency chains.

7. Recommend practical management actions.

8. Do not resurrect risks from previous analyses.

9. If important information is unavailable,
   include it in information_gaps.

10. Return only the structure required by the
    JSON schema.
"""


    # --------------------------------------------------------
    # 4. CALL OPENAI
    # --------------------------------------------------------

    response = client.responses.create(

        model=AI_MODEL,

        instructions=instructions,

        input=f"""
Analyze the following NovaBridge project portfolio.

PROJECT DATA:

{ai_input_json}
""",

        text={
            "format": {
                "type": "json_schema",
                "name":
                    "novabridge_risk_analysis",
                "strict": True,
                "schema":
                    risk_schema
            }
        }
    )


    # --------------------------------------------------------
    # 5. CONVERT AI RESPONSE TO PYTHON DATA
    # --------------------------------------------------------

    ai_analysis = json.loads(
        response.output_text
    )


    # --------------------------------------------------------
    # 6. RETURN BOTH INPUT + RESULT
    # --------------------------------------------------------

    return {

        "ai_input_data":
            ai_input_data,

        "ai_analysis":
            ai_analysis
    }

In [ ]:
# ============================================================
# CELL 8 - AI VALIDATION & GROUNDING ENGINE
# ============================================================


def validate_and_ground_ai(
    data,
    engine_results,
    ai_result
):

    ai_analysis = ai_result[
        "ai_analysis"
    ]

    task_lookup = data[
        "task_lookup"
    ]

    project_lookup = data[
        "project_lookup"
    ]


    # --------------------------------------------------------
    # 1. VALIDATE AI REFERENCES
    # --------------------------------------------------------

    validation_results = []


    for risk in ai_analysis[
        "top_risks"
    ]:

        project_id = risk[
            "project_id"
        ]

        task_id = risk[
            "root_task_id"
        ]


        project_exists = (
            project_id
            in project_lookup
        )


        task_exists = (
            task_id
            in task_lookup
        )


        task_matches_project = False


        if task_exists:

            task_matches_project = (

                task_lookup[
                    task_id
                ]["project_id"]

                == project_id
            )


        has_recommended_actions = (

            len(
                risk[
                    "recommended_actions"
                ]
            ) > 0
        )


        validation_passed = (

            project_exists

            and task_exists

            and task_matches_project

            and has_recommended_actions
        )


        validation_results.append({

            "project_id":
                project_id,

            "task_id":
                task_id,

            "project_exists":
                project_exists,

            "task_exists":
                task_exists,

            "task_matches_project":
                task_matches_project,

            "has_recommended_actions":
                has_recommended_actions,

            "validation_passed":
                validation_passed
        })


    # --------------------------------------------------------
    # 2. BUILD PYTHON SOURCE-OF-TRUTH RISK SET
    # --------------------------------------------------------

    deterministic_risk_task_ids = set()


    for risk in engine_results[
        "dependency_risks"
    ]:

        if "task_id" in risk:

            deterministic_risk_task_ids.add(
                risk["task_id"]
            )


    for risk in engine_results[
        "dependency_chain_risks"
    ]:

        if "root_task_id" in risk:

            deterministic_risk_task_ids.add(
                risk[
                    "root_task_id"
                ]
            )


    # --------------------------------------------------------
    # 3. GROUND AI RISKS AGAINST PYTHON RESULTS
    # --------------------------------------------------------

    grounding_results = []


    for risk in ai_analysis[
        "top_risks"
    ]:

        task_id = risk[
            "root_task_id"
        ]


        grounded = (

            task_id
            in deterministic_risk_task_ids
        )


        grounding_results.append({

            "project_id":
                risk[
                    "project_id"
                ],

            "task_id":
                task_id,

            "grounded_in_python":
                grounded
        })


    # --------------------------------------------------------
    # 4. CALCULATE SCORES
    # --------------------------------------------------------

    total_ai_risks = len(
        ai_analysis[
            "top_risks"
        ]
    )


    validated_risks = sum(

        1

        for result
        in validation_results

        if result[
            "validation_passed"
        ]
    )


    grounded_risks = sum(

        1

        for result
        in grounding_results

        if result[
            "grounded_in_python"
        ]
    )


    if total_ai_risks > 0:

        validation_score = (

            validated_risks
            / total_ai_risks
            * 100
        )


        grounding_score = (

            grounded_risks
            / total_ai_risks
            * 100
        )

    else:

        validation_score = 0

        grounding_score = 0


    # --------------------------------------------------------
    # 5. CHECK REQUIRED STRUCTURE
    # --------------------------------------------------------

    required_fields = [

        "portfolio_health",

        "executive_summary",

        "top_risks",

        "information_gaps"
    ]


    structure_passed = all(

        field in ai_analysis

        for field
        in required_fields
    )


    # --------------------------------------------------------
    # 6. OVERALL EVALUATION
    # --------------------------------------------------------

    if (

        validation_score == 100

        and grounding_score == 100

        and structure_passed

    ):

        overall_evaluation = (
            "PASS"
        )

    else:

        overall_evaluation = (
            "REVIEW REQUIRED"
        )


    # --------------------------------------------------------
    # 7. CREATE SCORECARD
    # --------------------------------------------------------

    evaluation_scorecard = {

        "total_ai_risks":
            total_ai_risks,

        "validated_risks":
            validated_risks,

        "grounded_risks":
            grounded_risks,

        "validation_score_percent":
            round(
                validation_score,
                2
            ),

        "grounding_score_percent":
            round(
                grounding_score,
                2
            ),

        "structured_output_valid":
            structure_passed,

        "overall_evaluation":
            overall_evaluation
    }


    # --------------------------------------------------------
    # 8. RETURN QUALITY-CONTROL RESULTS
    # --------------------------------------------------------

    return {

        "validation_results":
            validation_results,

        "grounding_results":
            grounding_results,

        "deterministic_risk_task_ids":
            sorted(
                deterministic_risk_task_ids
            ),

        "evaluation_scorecard":
            evaluation_scorecard
    }

In [ ]:
# ============================================================
# CELL 9 - PM INTELLIGENCE REPORT GENERATOR
# ============================================================


def generate_pm_report(
    engine_results,
    ai_result,
    quality_results,
    analysis_date
):

    ai_analysis = ai_result[
        "ai_analysis"
    ]

    scorecard = quality_results[
        "evaluation_scorecard"
    ]


    # --------------------------------------------------------
    # REPORT HEADER
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "        NOVABRIDGE PM INTELLIGENCE REPORT"
    )

    print(
        "============================================================"
    )


    print("\nAnalysis Date:")

    print(
        analysis_date.strftime(
            "%Y-%m-%d"
        )
    )


    print("\nPortfolio Health:")

    print(
        ai_analysis[
            "portfolio_health"
        ]
    )


    # --------------------------------------------------------
    # EXECUTIVE SUMMARY
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "EXECUTIVE SUMMARY"
    )

    print(
        "============================================================\n"
    )


    print(
        ai_analysis[
            "executive_summary"
        ]
    )


    # --------------------------------------------------------
    # PROJECT HEALTH
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "PROJECT HEALTH"
    )

    print(
        "============================================================"
    )


    for project in engine_results[
        "project_summaries"
    ]:

        print(
            "\n"
            + project["project_id"]
            + " - "
            + project["project_name"]
        )


        print(
            "Health:",
            project[
                "project_health"
            ]
        )


        print(
            "Completion:",
            str(
                project[
                    "completion_percent"
                ]
            ) + "%"
        )


        print(
            "Completed Tasks:",
            project[
                "completed_tasks"
            ],
            "/",
            project[
                "total_tasks"
            ]
        )


        print(
            "Delayed Tasks:",
            project[
                "delayed_tasks"
            ]
        )


        print(
            "Overdue Tasks:",
            project[
                "overdue_tasks"
            ]
        )


    # --------------------------------------------------------
    # TOP MANAGEMENT RISKS
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "TOP MANAGEMENT RISKS"
    )

    print(
        "============================================================"
    )


    if ai_analysis["top_risks"]:

        for number, risk in enumerate(
            ai_analysis[
                "top_risks"
            ],
            start=1
        ):

            print(
                "\nRISK",
                number
            )

            print(
                "------------------------------------------------------------"
            )


            print(
                "Project:",
                risk["project_id"],
                "-",
                risk["project_name"]
            )


            print(
                "Root Task:",
                risk[
                    "root_task_id"
                ]
            )


            print(
                "Risk:",
                risk[
                    "risk_title"
                ]
            )


            print(
                "Management Attention:",
                risk[
                    "management_attention"
                ]
            )


            print(
                "\nBusiness Impact:"
            )


            print(
                risk[
                    "business_impact"
                ]
            )


            print(
                "\nRecommended Actions:"
            )


            for action_number, action in enumerate(
                risk[
                    "recommended_actions"
                ],
                start=1
            ):

                print(
                    str(
                        action_number
                    ) + ".",
                    action
                )

    else:

        print(
            "\nNo major AI-prioritized risks."
        )


    # --------------------------------------------------------
    # PYTHON DEPENDENCY ANALYSIS
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "DETERMINISTIC DEPENDENCY ANALYSIS"
    )

    print(
        "============================================================"
    )


    chain_risks = engine_results[
        "dependency_chain_risks"
    ]


    if chain_risks:

        for risk in chain_risks:

            print(
                "\nRoot Task:",
                risk[
                    "root_task_id"
                ],
                "-",
                risk[
                    "root_task_name"
                ]
            )


            print(
                "Chain Impact:",
                risk[
                    "chain_impact"
                ]
            )


            print(
                "Impact Score:",
                risk[
                    "chain_impact_score"
                ]
            )


            print(
                "Affected Downstream Tasks:",
                ", ".join(
                    risk[
                        "downstream_tasks"
                    ]
                )
            )


            print(
                "Critical Downstream Tasks:",
                risk[
                    "critical_downstream_tasks"
                ]
            )

    else:

        print(
            "\nNo dependency-chain risks detected."
        )


    # --------------------------------------------------------
    # INFORMATION GAPS
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "INFORMATION GAPS"
    )

    print(
        "============================================================"
    )


    gaps = ai_analysis[
        "information_gaps"
    ]


    if gaps:

        for number, gap in enumerate(
            gaps,
            start=1
        ):

            print(
                str(number) + ".",
                gap
            )

    else:

        print(
            "\nNo major information gaps reported."
        )


    # --------------------------------------------------------
    # AI QUALITY CONTROL
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "AI QUALITY CONTROL"
    )

    print(
        "============================================================"
    )


    print(
        "\nAI Risks Generated:",
        scorecard[
            "total_ai_risks"
        ]
    )


    print(
        "Validated Risks:",
        scorecard[
            "validated_risks"
        ]
    )


    print(
        "Grounded Risks:",
        scorecard[
            "grounded_risks"
        ]
    )


    print(
        "Validation Score:",
        str(
            scorecard[
                "validation_score_percent"
            ]
        ) + "%"
    )


    print(
        "Grounding Score:",
        str(
            scorecard[
                "grounding_score_percent"
            ]
        ) + "%"
    )


    print(
        "Structured Output Valid:",
        scorecard[
            "structured_output_valid"
        ]
    )


    print(
        "Overall Evaluation:",
        scorecard[
            "overall_evaluation"
        ]
    )


    # --------------------------------------------------------
    # END REPORT
    # --------------------------------------------------------

    print(
        "\n============================================================"
    )

    print(
        "END OF NOVABRIDGE PM INTELLIGENCE REPORT"
    )

    print(
        "============================================================"
    )

In [ ]:
# ============================================================
# CELL 10 - MASTER NOVABRIDGE PIPELINE
# ============================================================


def run_novabridge_system(
    excel_file
):

    print(
        "\n============================================================"
    )

    print(
        "STARTING NOVABRIDGE AI PROJECT MANAGEMENT SYSTEM"
    )

    print(
        "============================================================"
    )


    # --------------------------------------------------------
    # 1. DETERMINE ANALYSIS DATE
    # --------------------------------------------------------

    if ANALYSIS_DATE_OVERRIDE:

        analysis_date = datetime.strptime(
            ANALYSIS_DATE_OVERRIDE,
            "%Y-%m-%d"
        )

    else:

        analysis_date = datetime.now()


    print(
        "\nAnalysis Date:",
        analysis_date.strftime(
            "%Y-%m-%d"
        )
    )


    # --------------------------------------------------------
    # 2. LOAD + VALIDATE EXCEL
    # --------------------------------------------------------

    print(
        "\n[1/6] Loading and validating Excel..."
    )

    df = load_and_validate_excel(
        excel_file
    )


    # --------------------------------------------------------
    # 3. BUILD INTERNAL DATA
    # --------------------------------------------------------

    print(
        "\n[2/6] Building NovaBridge data structures..."
    )

    data = build_novabridge_data(
        df
    )

    print(
        "✅ Data structures created."
    )


    # --------------------------------------------------------
    # 4. RUN DETERMINISTIC ENGINE
    # --------------------------------------------------------

    print(
        "\n[3/6] Running deterministic risk engine..."
    )

    engine_results = run_deterministic_engine(
        data,
        analysis_date
    )

    print(
        "✅ Deterministic analysis complete."
    )


    # --------------------------------------------------------
    # 5. RUN AI RISK ANALYST
    # --------------------------------------------------------

    print(
        "\n[4/6] Running AI Risk Analyst..."
    )

    ai_result = run_ai_risk_analyst(
        data,
        engine_results,
        analysis_date
    )

    print(
        "✅ AI analysis complete."
    )


    # --------------------------------------------------------
    # 6. VALIDATE + GROUND AI
    # --------------------------------------------------------

    print(
        "\n[5/6] Validating and grounding AI output..."
    )

    quality_results = validate_and_ground_ai(
        data,
        engine_results,
        ai_result
    )

    print(
        "✅ AI quality-control checks complete."
    )


    # --------------------------------------------------------
    # 7. GENERATE REPORT
    # --------------------------------------------------------

    print(
        "\n[6/6] Generating PM Intelligence Report..."
    )


    generate_pm_report(
        engine_results,
        ai_result,
        quality_results,
        analysis_date
    )


    # --------------------------------------------------------
    # 8. PACKAGE COMPLETE RESULT
    # --------------------------------------------------------

    result = {

        "analysis_date":
            analysis_date.strftime(
                "%Y-%m-%d"
            ),

        "source_file":
            excel_file,

        "data":
            data,

        "engine_results":
            engine_results,

        "ai_result":
            ai_result,

        "quality_results":
            quality_results
    }


    # --------------------------------------------------------
    # 9. RETURN RESULT
    # --------------------------------------------------------

    return result

In [ ]:
# Restore NovaBridge configuration

ANALYSIS_DATE_OVERRIDE = None

print("✅ ANALYSIS_DATE_OVERRIDE restored.")

In [ ]:
# Restore required Python imports

from datetime import datetime

print("✅ datetime import restored.")

In [ ]:
# ============================================================
# CELL 11 - RUN NOVABRIDGE
# ============================================================


EXCEL_FILE = (
    "novabridge_fresh_project_data.xlsx"
)


result = run_novabridge_system(
    EXCEL_FILE
)

In [ ]:
# ============================================================
# CELL 12 - EXPORT NOVABRIDGE RESULTS
# CORRECTED VERSION
# ============================================================

import json
from datetime import datetime


print("=" * 55)
print("EXPORTING NOVABRIDGE RESULTS")
print("=" * 55)


# ------------------------------------------------------------
# 1. CREATE EXPORT TIMESTAMP
# ------------------------------------------------------------

export_timestamp = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)


# ------------------------------------------------------------
# 2. GET RESULTS FROM MASTER PIPELINE
# ------------------------------------------------------------

ai_analysis = (
    result["ai_result"]["ai_analysis"]
)

engine_results = (
    result["engine_results"]
)

quality_results = (
    result["quality_results"]
)

scorecard = (
    quality_results[
        "evaluation_scorecard"
    ]
)


# ------------------------------------------------------------
# 3. EXPORT COMPLETE STRUCTURED RESULT AS JSON
# ------------------------------------------------------------

json_filename = (
    f"novabridge_analysis_{export_timestamp}.json"
)


with open(
    json_filename,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        result,
        file,
        indent=2,
        default=str
    )


print("\n✅ Structured JSON exported:")
print(json_filename)


# ------------------------------------------------------------
# 4. CREATE HUMAN-READABLE REPORT
# ------------------------------------------------------------

report_lines = []


report_lines.append("=" * 65)

report_lines.append(
    "NOVABRIDGE AI PROJECT MANAGEMENT SYSTEM"
)

report_lines.append(
    "PM INTELLIGENCE REPORT"
)

report_lines.append("=" * 65)


report_lines.append("")

report_lines.append(
    f"Analysis Date: "
    f"{result['analysis_date']}"
)

report_lines.append(
    f"Source File: "
    f"{result['source_file']}"
)


# ------------------------------------------------------------
# 5. PORTFOLIO HEALTH
# ------------------------------------------------------------

report_lines.append("")

report_lines.append(
    "PORTFOLIO HEALTH"
)

report_lines.append("-" * 65)


report_lines.append(
    ai_analysis[
        "portfolio_health"
    ]
)


# ------------------------------------------------------------
# 6. EXECUTIVE SUMMARY
# ------------------------------------------------------------

report_lines.append("")

report_lines.append(
    "EXECUTIVE SUMMARY"
)

report_lines.append("-" * 65)


report_lines.append(
    ai_analysis[
        "executive_summary"
    ]
)


# ------------------------------------------------------------
# 7. PROJECT HEALTH
# ------------------------------------------------------------

report_lines.append("")

report_lines.append(
    "PROJECT HEALTH"
)

report_lines.append("-" * 65)


for project in engine_results[
    "project_summaries"
]:

    report_lines.append("")

    report_lines.append(
        f"{project['project_id']} - "
        f"{project['project_name']}"
    )

    report_lines.append(
        f"Health: "
        f"{project['project_health']}"
    )

    report_lines.append(
        f"Completion: "
        f"{project['completion_percent']}%"
    )

    report_lines.append(
        f"Completed Tasks: "
        f"{project['completed_tasks']} / "
        f"{project['total_tasks']}"
    )

    report_lines.append(
        f"Delayed Tasks: "
        f"{project['delayed_tasks']}"
    )

    report_lines.append(
        f"Overdue Tasks: "
        f"{project['overdue_tasks']}"
    )


# ------------------------------------------------------------
# 8. TOP MANAGEMENT RISKS
# ------------------------------------------------------------

report_lines.append("")

report_lines.append(
    "TOP MANAGEMENT RISKS"
)

report_lines.append("-" * 65)


if ai_analysis["top_risks"]:

    for number, risk in enumerate(
        ai_analysis["top_risks"],
        start=1
    ):

        report_lines.append("")

        report_lines.append(
            f"RISK {number}"
        )

        report_lines.append(
            f"Project: "
            f"{risk['project_id']} - "
            f"{risk['project_name']}"
        )

        report_lines.append(
            f"Root Task: "
            f"{risk['root_task_id']}"
        )

        report_lines.append(
            f"Risk: "
            f"{risk['risk_title']}"
        )

        report_lines.append(
            f"Management Attention: "
            f"{risk['management_attention']}"
        )

        report_lines.append("")

        report_lines.append(
            "Business Impact:"
        )

        report_lines.append(
            risk["business_impact"]
        )

        report_lines.append("")

        report_lines.append(
            "Recommended Actions:"
        )


        for action_number, action in enumerate(
            risk["recommended_actions"],
            start=1
        ):

            report_lines.append(
                f"{action_number}. {action}"
            )

else:

    report_lines.append(
        "No major AI-prioritized risks."
    )


# ------------------------------------------------------------
# 9. DETERMINISTIC DEPENDENCY RISKS
# ------------------------------------------------------------

report_lines.append("")

report_lines.append(
    "DEPENDENCY-CHAIN ANALYSIS"
)

report_lines.append("-" * 65)


chain_risks = engine_results[
    "dependency_chain_risks"
]


if chain_risks:

    for risk in chain_risks:

        report_lines.append("")

        report_lines.append(
            f"Root Task: "
            f"{risk['root_task_id']} - "
            f"{risk['root_task_name']}"
        )

        report_lines.append(
            f"Chain Impact: "
            f"{risk['chain_impact']}"
        )

        report_lines.append(
            f"Impact Score: "
            f"{risk['chain_impact_score']}"
        )

        report_lines.append(
            "Affected Downstream Tasks: "
            + ", ".join(
                risk["downstream_tasks"]
            )
        )

        report_lines.append(
            f"Critical Downstream Tasks: "
            f"{risk['critical_downstream_tasks']}"
        )

else:

    report_lines.append(
        "No dependency-chain risks detected."
    )


# ------------------------------------------------------------
# 10. INFORMATION GAPS
# ------------------------------------------------------------

report_lines.append("")

report_lines.append(
    "INFORMATION GAPS"
)

report_lines.append("-" * 65)


gaps = ai_analysis[
    "information_gaps"
]


if gaps:

    for number, gap in enumerate(
        gaps,
        start=1
    ):

        report_lines.append(
            f"{number}. {gap}"
        )

else:

    report_lines.append(
        "No major information gaps reported."
    )


# ------------------------------------------------------------
# 11. AI QUALITY CONTROL
# ------------------------------------------------------------

report_lines.append("")

report_lines.append(
    "AI QUALITY CONTROL"
)

report_lines.append("-" * 65)


report_lines.append(
    f"AI Risks Generated: "
    f"{scorecard['total_ai_risks']}"
)

report_lines.append(
    f"Validated Risks: "
    f"{scorecard['validated_risks']}"
)

report_lines.append(
    f"Grounded Risks: "
    f"{scorecard['grounded_risks']}"
)

report_lines.append(
    f"Validation Score: "
    f"{scorecard['validation_score_percent']}%"
)

report_lines.append(
    f"Grounding Score: "
    f"{scorecard['grounding_score_percent']}%"
)

report_lines.append(
    f"Structured Output Valid: "
    f"{scorecard['structured_output_valid']}"
)

report_lines.append(
    f"Overall Evaluation: "
    f"{scorecard['overall_evaluation']}"
)


# ------------------------------------------------------------
# 12. SAVE TEXT REPORT
# ------------------------------------------------------------

report_filename = (
    f"novabridge_pm_report_"
    f"{export_timestamp}.txt"
)


with open(
    report_filename,
    "w",
    encoding="utf-8"
) as file:

    file.write(
        "\n".join(
            report_lines
        )
    )


print("\n✅ PM Intelligence Report exported:")
print(report_filename)


# ------------------------------------------------------------
# 13. FINAL CONFIRMATION
# ------------------------------------------------------------

print("\n" + "=" * 55)

print(
    "NOVABRIDGE EXPORT COMPLETE"
)

print("=" * 55)


print("\nFiles created:")

print(
    "1.",
    json_filename
)

print(
    "2.",
    report_filename
)


print(
    "\n✅ Results are ready for workflow automation."
)

In [ ]:
# ============================================================
# PHASE 3 - WORKFLOW AUTOMATION
# STEP 2 - PREPARE REAL NOVABRIDGE RISK FOR N8N
# CORRECTED FOR PRODUCTION PIPELINE
# ============================================================

# Get the real AI analysis from the production pipeline
ai_analysis = result["ai_result"]["ai_analysis"]

# Find risks that require IMMEDIATE management attention
immediate_risks = [
    risk
    for risk in ai_analysis["top_risks"]
    if risk["management_attention"] == "IMMEDIATE"
]

print("Immediate risks found:", len(immediate_risks))


if immediate_risks:

    # For now, use the highest-priority immediate risk
    top_risk = immediate_risks[0]

    n8n_payload = {
        "portfolio_health": ai_analysis["portfolio_health"],
        "project_id": top_risk["project_id"],
        "project_name": top_risk["project_name"],
        "top_risk_task": top_risk["root_task_id"],
        "risk_title": top_risk["risk_title"],
        "business_impact": top_risk["business_impact"],
        "management_attention": top_risk["management_attention"],
        "recommended_actions": top_risk["recommended_actions"]
    }

    print("\n✅ Real NovaBridge risk prepared for n8n:")
    print(n8n_payload)

else:

    n8n_payload = None

    print(
        "\n✅ No IMMEDIATE risk found. "
        "No escalation required."
    )

In [ ]:
from google.colab import userdata
import os

# Securely load n8n production webhook
os.environ["N8N_PRODUCTION_WEBHOOK_URL"] = userdata.get(
    "N8N_PRODUCTION_WEBHOOK_URL"
)

print("✅ n8n webhook loaded securely.")

In [ ]:
if os.getenv("N8N_PRODUCTION_WEBHOOK_URL"):
    print("✅ n8n production webhook detected.")
else:
    print("❌ n8n production webhook not detected.")

In [ ]:
# ============================================================
# PHASE 3 - STEP 4
# AUTOMATIC N8N RISK ESCALATION FUNCTION
# ============================================================

import requests

import os

N8N_PRODUCTION_WEBHOOK_URL = os.getenv(
    "N8N_PRODUCTION_WEBHOOK_URL"
)

if not N8N_PRODUCTION_WEBHOOK_URL:
    raise ValueError(
        "N8N_PRODUCTION_WEBHOOK_URL is not configured."
    )


def send_immediate_risk_to_n8n(ai_analysis):

    immediate_risks = [
        risk
        for risk in ai_analysis["top_risks"]
        if risk["management_attention"] == "IMMEDIATE"
    ]

    print(
        f"Immediate risks detected: {len(immediate_risks)}"
    )

    if len(immediate_risks) == 0:

        print(
            "✅ No IMMEDIATE risks. "
            "No escalation email required."
        )

        return None

    # For now, send the highest-priority immediate risk
    top_risk = immediate_risks[0]

    n8n_payload = {
        "portfolio_health":
            ai_analysis["portfolio_health"],

        "project_id":
            top_risk["project_id"],

        "project_name":
            top_risk["project_name"],

        "top_risk_task":
            top_risk["root_task_id"],

        "risk_title":
            top_risk["risk_title"],

        "business_impact":
            top_risk["business_impact"],

        "management_attention":
            top_risk["management_attention"],

        "recommended_actions":
            top_risk["recommended_actions"]
    }

    response = requests.post(
        N8N_PRODUCTION_WEBHOOK_URL,
        json=n8n_payload,
        timeout=30
    )

    print(
        "n8n Status Code:",
        response.status_code
    )

    if response.status_code == 200:
        print(
            "✅ NovaBridge escalation sent automatically."
        )
    else:
        print(
            "❌ n8n escalation failed."
        )

    return response

In [ ]:
# ============================================================
# PHASE 3 - STEP 6
# ONE-RUN NOVABRIDGE AUTOMATION WRAPPER
# ============================================================

def run_novabridge_automated(excel_file):

    # 1. Run the full NovaBridge analysis
    result = run_novabridge_system(
        excel_file
    )

    # 2. Get the AI analysis
    ai_analysis = result["ai_result"]["ai_analysis"]

    # 3. Automatically send any IMMEDIATE risk to n8n
    send_immediate_risk_to_n8n(
        ai_analysis
    )

    # 4. Return the complete result
    return result


print("✅ Automated NovaBridge wrapper created.")

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 1 - CHECK OUR PRODUCTION SYSTEM
# ============================================================

print("🤖 NovaBridge AI PM Copilot")
print("=" * 55)

print("\nChecking production result...")

print("Result type:")
print(type(result))

print("\nAvailable result sections:")
print(result.keys())

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 2 - CREATE READ-ONLY PM TOOLS
# ============================================================

def get_portfolio_health():
    """
    Returns the AI-assessed health of the NovaBridge portfolio.
    """
    ai_analysis = result["ai_result"]["ai_analysis"]

    return {
        "portfolio_health": ai_analysis["portfolio_health"],
        "executive_summary": ai_analysis["executive_summary"]
    }


def get_top_risks():
    """
    Returns the current top project risks.
    """
    ai_analysis = result["ai_result"]["ai_analysis"]

    return ai_analysis["top_risks"]


def get_project_status(project_id):
    """
    Returns project health information for one project.
    Example: P001, P002 or P003
    """

    project_health = result["engine_results"]["project_summaries"]

    for project in project_health:

        if project["project_id"] == project_id:
            return project

    return {
        "error": f"Project {project_id} was not found."
    }


def get_task_details(task_id):
    """
    Returns the original project data for one task.
    Example: T008
    """

    tasks = result["data"]["tasks"]

    for task in tasks:

        if task["task_id"] == task_id:
            return task

    return {
        "error": f"Task {task_id} was not found."
    }


print("✅ NovaBridge PM Copilot tools created.")

In [ ]:
get_project_status("P002")

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 3 - TEST PM TOOLS
# ============================================================

print("TEST 1 - PORTFOLIO HEALTH")
print("-" * 50)
print(get_portfolio_health())


print("\nTEST 2 - TOP RISKS")
print("-" * 50)
print(get_top_risks())


print("\nTEST 3 - PROJECT P002")
print("-" * 50)
print(get_project_status("P002"))


print("\nTEST 4 - TASK T008")
print("-" * 50)
print(get_task_details("T008"))

In [ ]:
# ============================================================
# PHASE 4 - DEBUG PROJECT STATUS TOOL
# CHECK REAL ENGINE RESULT STRUCTURE
# ============================================================

print("ENGINE RESULT KEYS:")
print(result["engine_results"].keys())

print("\nDATA KEYS:")
print(result["data"].keys())

In [ ]:
# ============================================================
# PHASE 4 - FIX PROJECT STATUS TOOL
# ============================================================

def get_project_status(project_id):
    """
    Returns project health information for one project.
    Example: P001, P002 or P003
    """

    project_summaries = (
        result["engine_results"]["project_summaries"]
    )

    for project in project_summaries:

        if project["project_id"] == project_id:
            return project

    return {
        "error": f"Project {project_id} was not found."
    }


print("✅ get_project_status() fixed.")

In [ ]:
print(get_project_status("P002"))
print()
print(get_task_details("T008"))

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 4 - DEFINE TOOLS FOR OPENAI
# ============================================================

import json

copilot_tools = [
    {
        "type": "function",
        "name": "get_portfolio_health",
        "description": "Get the overall NovaBridge portfolio health and executive summary.",
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "get_top_risks",
        "description": "Get the current highest-priority project risks.",
        "parameters": {
            "type": "object",
            "properties": {},
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "get_project_status",
        "description": "Get status and health information for a specific project.",
        "parameters": {
            "type": "object",
            "properties": {
                "project_id": {
                    "type": "string",
                    "description": "NovaBridge project ID such as P001, P002, or P003."
                }
            },
            "required": ["project_id"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "get_task_details",
        "description": "Get detailed information about a specific task.",
        "parameters": {
            "type": "object",
            "properties": {
                "task_id": {
                    "type": "string",
                    "description": "NovaBridge task ID such as T008."
                }
            },
            "required": ["task_id"],
            "additionalProperties": False
        }
    }
]

print("✅ OpenAI tool definitions created.")

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 5 - CREATE THE TOOL-CALLING COPILOT
# ============================================================

import json

def ask_novabridge_copilot(user_question):

    print("🤖 NovaBridge AI PM Copilot")
    print("-" * 55)
    print("Question:", user_question)

    # Step 1: Ask the model what it needs
    response = client.responses.create(
        model="gpt-5.6-luna",
        instructions="""
You are the NovaBridge AI Project Management Copilot.

Use the available project-management tools whenever they are
needed to answer the user's question.

Do not invent project facts.

Use tool results as the source of truth.

Give concise, professional, project-manager-friendly answers.
""",
        input=user_question,
        tools=copilot_tools
    )

    # Step 2: Look for a function/tool call
    tool_calls = [
        item
        for item in response.output
        if item.type == "function_call"
    ]

    if not tool_calls:
        print("\nCopilot:")
        print(response.output_text)
        return response.output_text

    # For our first version, process the first tool call
    tool_call = tool_calls[0]

    tool_name = tool_call.name
    arguments = json.loads(tool_call.arguments)

    print("\n🔧 Tool selected:", tool_name)
    print("Arguments:", arguments)

    # Step 3: Run the corresponding Python function
    if tool_name == "get_portfolio_health":
        tool_result = get_portfolio_health()

    elif tool_name == "get_top_risks":
        tool_result = get_top_risks()

    elif tool_name == "get_project_status":
        tool_result = get_project_status(
            arguments["project_id"]
        )

    elif tool_name == "get_task_details":
        tool_result = get_task_details(
            arguments["task_id"]
        )

    else:
        tool_result = {
            "error": f"Unknown tool: {tool_name}"
        }

    print("\n📊 Tool result received.")

    # Step 4: Give the tool result back to the model
    final_response = client.responses.create(
        model="gpt-5.6-luna",
        previous_response_id=response.id,
        input=[
            {
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": json.dumps(
                    tool_result,
                    default=str
                )
            }
        ]
    )

    print("\nCopilot:")
    print(final_response.output_text)

    return final_response.output_text


print("✅ NovaBridge AI PM Copilot created.")

In [ ]:
ask_novabridge_copilot(
    "What is the current status of project P002?"
)

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 6 - FIRST AGENT TEST
# ============================================================

ask_novabridge_copilot(
    "What is happening with project P002?"
)

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 7 - MULTI-TOOL AGENT LOOP
# ============================================================

import json


def ask_novabridge_agent(user_question):

    print("🤖 NovaBridge AI PM Agent")
    print("-" * 55)
    print("Question:", user_question)

    instructions = """
You are the NovaBridge AI Project Management Copilot.

Use the available project-management tools whenever needed.

Rules:
1. Do not invent project facts.
2. Treat tool results as the source of truth.
3. You may call multiple tools if needed.
4. Use project IDs and task IDs exactly as provided.
5. Give concise, professional, project-manager-friendly answers.
"""

    # --------------------------------------------------------
    # STEP 1 - Send the user's question to the model
    # --------------------------------------------------------

    response = client.responses.create(
        model="gpt-5.6-luna",
        instructions=instructions,
        input=user_question,
        tools=copilot_tools
    )

    # Prevent an accidental endless loop
    max_tool_rounds = 5
    round_number = 0

    # --------------------------------------------------------
    # STEP 2 - Keep working until the AI has a final answer
    # --------------------------------------------------------

    while round_number < max_tool_rounds:

        round_number += 1

        tool_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        # ----------------------------------------------------
        # If there are no more tool calls, we have the answer
        # ----------------------------------------------------

        if not tool_calls:

            print("\n✅ Final Copilot Answer:")
            print(response.output_text)

            return response.output_text

        print(
            f"\n🔄 Tool round {round_number}: "
            f"{len(tool_calls)} tool call(s)"
        )

        tool_outputs = []

        # ----------------------------------------------------
        # STEP 3 - Execute every tool requested by the AI
        # ----------------------------------------------------

        for tool_call in tool_calls:

            tool_name = tool_call.name
            arguments = json.loads(
                tool_call.arguments
            )

            print("\n🔧 Tool selected:", tool_name)
            print("Arguments:", arguments)

            if tool_name == "get_portfolio_health":

                tool_result = get_portfolio_health()

            elif tool_name == "get_top_risks":

                tool_result = get_top_risks()

            elif tool_name == "get_project_status":

                tool_result = get_project_status(
                    arguments["project_id"]
                )

            elif tool_name == "get_task_details":

                tool_result = get_task_details(
                    arguments["task_id"]
                )

            else:

                tool_result = {
                    "error": f"Unknown tool: {tool_name}"
                }

            print("📊 Tool result received.")

            tool_outputs.append(
                {
                    "type": "function_call_output",
                    "call_id": tool_call.call_id,
                    "output": json.dumps(
                        tool_result,
                        default=str
                    )
                }
            )

        # ----------------------------------------------------
        # STEP 4 - Give ALL tool results back to the model
        # ----------------------------------------------------

        response = client.responses.create(
            model="gpt-5.6-luna",
            instructions=instructions,
            previous_response_id=response.id,
            input=tool_outputs,
            tools=copilot_tools
        )

    print(
        "\n⚠️ Agent stopped because the maximum "
        "number of tool rounds was reached."
    )

    return response.output_text


print("✅ NovaBridge multi-tool AI agent created.")

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 8 - TEST MULTI-TOOL REASONING
# ============================================================

agent_answer = ask_novabridge_agent(
    """
    What is the overall portfolio health?
    Identify the biggest current risk.
    Then give me the detailed task information for the task
    causing that risk.
    """
)

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 7 - TEST INTELLIGENT TOOL SELECTION
# ============================================================

ask_novabridge_copilot(
    "What are the biggest risks in the portfolio?"
)

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 8 - MULTI-TOOL AGENT LOOP
# ============================================================

import json

def ask_novabridge_copilot_v2(user_question):

    print("🤖 NovaBridge AI PM Copilot V2")
    print("-" * 60)
    print("Question:", user_question)

    response = client.responses.create(
        model="gpt-5.6-luna",
        instructions="""
You are the NovaBridge AI Project Management Copilot.

Use the available project-management tools whenever needed.

You may call multiple tools if the question requires information
from more than one project, task, or portfolio section.

Do not invent project facts.
Use tool results as the source of truth.

Give concise, professional, project-manager-friendly answers.
""",
        input=user_question,
        tools=copilot_tools
    )

    max_rounds = 5
    round_number = 0

    while round_number < max_rounds:

        round_number += 1

        tool_calls = [
            item
            for item in response.output
            if item.type == "function_call"
        ]

        # If the model no longer needs a tool,
        # we have the final answer.
        if not tool_calls:

            print("\nCopilot:")
            print(response.output_text)

            return response.output_text

        print(
            f"\n🔄 Tool round {round_number}"
        )

        tool_outputs = []

        for tool_call in tool_calls:

            tool_name = tool_call.name

            arguments = json.loads(
                tool_call.arguments
            )

            print(
                "\n🔧 Tool selected:",
                tool_name
            )

            print(
                "Arguments:",
                arguments
            )

            # --------------------------------------------
            # Run the correct NovaBridge Python tool
            # --------------------------------------------

            if tool_name == "get_portfolio_health":

                tool_result = (
                    get_portfolio_health()
                )

            elif tool_name == "get_top_risks":

                tool_result = (
                    get_top_risks()
                )

            elif tool_name == "get_project_status":

                tool_result = (
                    get_project_status(
                        arguments["project_id"]
                    )
                )

            elif tool_name == "get_task_details":

                tool_result = (
                    get_task_details(
                        arguments["task_id"]
                    )
                )

            else:

                tool_result = {
                    "error":
                    f"Unknown tool: {tool_name}"
                }

            print("📊 Tool result received.")

            tool_outputs.append(
                {
                    "type":
                        "function_call_output",

                    "call_id":
                        tool_call.call_id,

                    "output":
                        json.dumps(
                            tool_result,
                            default=str
                        )
                }
            )

        # --------------------------------------------
        # Give ALL tool results back to the model
        # --------------------------------------------

        response = client.responses.create(
            model="gpt-5.6-luna",
            previous_response_id=response.id,
            input=tool_outputs,
            tools=copilot_tools
        )

    print(
        "\n⚠️ Maximum tool rounds reached."
    )

    return response.output_text


print("✅ NovaBridge Multi-Tool Copilot created.")

In [ ]:
# ============================================================
# PHASE 4 - STEP 9
# TEST MULTI-TOOL REASONING
# ============================================================

ask_novabridge_copilot_v2(
    "Compare projects P002 and P003. "
    "Which one needs more management attention and why?"
)

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 10 - CONVERSATIONAL MEMORY
# ============================================================

import json


def start_novabridge_chat():

    print("🤖 NovaBridge AI PM Copilot - Conversation Started")
    print("-" * 60)

    # This stores the previous OpenAI response ID
    # so the conversation can continue across turns.
    conversation_response_id = None

    def chat(user_question):

        nonlocal conversation_response_id

        print("\n👤 You:", user_question)

        # ----------------------------------------------------
        # 1. Send the user's question to OpenAI
        # ----------------------------------------------------

        request_args = {
            "model": "gpt-5.6-luna",

            "instructions": """
You are the NovaBridge AI Project Management Copilot.

You are having an ongoing conversation with a project manager.

Remember the conversational context from previous turns.

Examples:
- "it" may refer to the previously discussed task or project.
- "that project" may refer to the previously discussed project.
- "that task" may refer to the previously discussed task.
- "its biggest risk" may refer to the project currently being discussed.

Use NovaBridge tools whenever project facts are required.

Do not invent project information.
Tool results are the source of truth.

Give concise, professional, actionable project-management answers.
""",

            "input": user_question,

            "tools": copilot_tools
        }

        # If we already have a conversation,
        # continue from the previous response.
        if conversation_response_id is not None:
            request_args["previous_response_id"] = (
                conversation_response_id
            )

        response = client.responses.create(
            **request_args
        )

        # ----------------------------------------------------
        # 2. Allow the agent to use tools
        # ----------------------------------------------------

        max_rounds = 5
        round_number = 0

        while round_number < max_rounds:

            round_number += 1

            tool_calls = [
                item
                for item in response.output
                if item.type == "function_call"
            ]

            # If the AI does not need another tool,
            # we have the final answer.
            if not tool_calls:

                conversation_response_id = response.id

                print("\n🤖 Copilot:")
                print(response.output_text)

                return response.output_text

            print(
                f"\n🔄 Tool round {round_number}"
            )

            tool_outputs = []

            for tool_call in tool_calls:

                tool_name = tool_call.name

                arguments = json.loads(
                    tool_call.arguments
                )

                print(
                    "\n🔧 Tool selected:",
                    tool_name
                )

                print(
                    "Arguments:",
                    arguments
                )

                # --------------------------------------------
                # Run the correct NovaBridge Python tool
                # --------------------------------------------

                if tool_name == "get_portfolio_health":

                    tool_result = (
                        get_portfolio_health()
                    )

                elif tool_name == "get_top_risks":

                    tool_result = (
                        get_top_risks()
                    )

                elif tool_name == "get_project_status":

                    tool_result = (
                        get_project_status(
                            arguments["project_id"]
                        )
                    )

                elif tool_name == "get_task_details":

                    tool_result = (
                        get_task_details(
                            arguments["task_id"]
                        )
                    )

                else:

                    tool_result = {
                        "error":
                        f"Unknown tool: {tool_name}"
                    }

                print(
                    "📊 Tool result received."
                )

                tool_outputs.append(
                    {
                        "type":
                            "function_call_output",

                        "call_id":
                            tool_call.call_id,

                        "output":
                            json.dumps(
                                tool_result,
                                default=str
                            )
                    }
                )

            # ------------------------------------------------
            # 3. Give all tool results back to OpenAI
            # ------------------------------------------------

            response = client.responses.create(
                model="gpt-5.6-luna",
                previous_response_id=response.id,
                input=tool_outputs,
                tools=copilot_tools
            )

        conversation_response_id = response.id

        print(
            "\n⚠️ Maximum tool rounds reached."
        )

        return response.output_text

    return chat


print("✅ Conversational NovaBridge Copilot created.")

In [ ]:
chat = start_novabridge_chat()

In [ ]:
chat("What is happening with project P002?")

In [ ]:
chat("What is its biggest risk?")

In [ ]:
chat("What is its biggest risk?")

In [ ]:
chat("Tell me about that task.")

In [ ]:
chat("Who owns it?")

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 11 - PM ACTION RECOMMENDATION TEST
# ============================================================

chat(
    "As the project manager, what should I do about this risk? "
    "Give me the 3 most important actions in priority order."
)

In [ ]:
# ============================================================
# PHASE 4 - AI PM COPILOT
# STEP 12 - HUMAN APPROVAL BEFORE ACTION
# ============================================================

def propose_escalation(task_id):

    task = get_task_details(task_id)

    if "error" in task:
        return task

    proposal = {
        "action": "SEND_ESCALATION",
        "task_id": task["task_id"],
        "project_id": task["project_id"],
        "project_name": task["project_name"],
        "task_name": task["task_name"],
        "owner": task["owner"],
        "status": task["status"],
        "priority": task["priority"],
        "latest_update": task["latest_update"],
        "approval_required": True
    }

    print("⚠️ ESCALATION PROPOSAL")
    print("-" * 55)

    print("Project:", proposal["project_id"], "-", proposal["project_name"])
    print("Task:", proposal["task_id"], "-", proposal["task_name"])
    print("Owner:", proposal["owner"])
    print("Status:", proposal["status"])
    print("Priority:", proposal["priority"])

    print("\nLatest Update:")
    print(proposal["latest_update"])

    print("\n🔒 Human approval required.")
    print("Nothing has been sent.")

    return proposal


print("✅ Human-approval action layer created.")

In [ ]:
# ============================================================
# PHASE 4 - STEP 12A
# TEST HUMAN APPROVAL LAYER
# ============================================================

approval_request = propose_escalation("T008")

In [ ]:
# ============================================================
# PHASE 4 - STEP 12B
# APPROVE / REJECT ESCALATION
# ============================================================

def handle_escalation_approval(
    approval_request,
    approved=False
):

    if approval_request is None:
        print("❌ No escalation proposal found.")
        return None

    if not approved:
        print("🛑 Escalation rejected.")
        print("Nothing was sent.")
        return {
            "status": "REJECTED",
            "sent": False
        }

    print("✅ Human approval received.")
    print("Sending escalation to n8n...")

    ai_analysis = result["ai_result"]["ai_analysis"]

    response = send_immediate_risk_to_n8n(
        ai_analysis
    )

    return {
        "status": "APPROVED",
        "sent": True,
        "response": response
    }


print("✅ Approval execution layer created.")

In [ ]:
# ============================================================
# PHASE 4 - STEP 12B
# TEST REJECTION PATH
# ============================================================

rejection_test = handle_escalation_approval(
    approval_request,
    approved=False
)

print("\nResult:")
print(rejection_test)

In [ ]:
# ============================================================
# PHASE 4 - OPTIONAL LIVE ESCALATION TEST
# DISABLED FOR PUBLIC GITHUB VERSION
# ============================================================
#
# This action can trigger the configured n8n workflow and send
# a real escalation. It is intentionally disabled in the public
# portfolio notebook.
#
# To test privately, configure your own Colab secret and
# deliberately uncomment the lines below.
#
# approval_test = handle_escalation_approval(
#     approval_request,
#     approved=True
# )
#
# print("\nResult:")
# print(approval_test)

print("🔒 Live escalation test disabled in public GitHub version.")


In [ ]:
# ============================================================
# PHASE 5 - PORTFOLIO DEMO
# STEP 1A - PRODUCTION SYSTEM HEALTH CHECK
# ============================================================

print("=" * 60)
print("NOVABRIDGE AI PROJECT MANAGEMENT SYSTEM")
print("PHASE 5 - PRODUCTION HEALTH CHECK")
print("=" * 60)

components = {
    "Main analysis pipeline":
        "run_novabridge_system" in globals(),

    "Automated workflow":
        "run_novabridge_automated" in globals(),

    "Project status tool":
        "get_project_status" in globals(),

    "Task details tool":
        "get_task_details" in globals(),

    "Portfolio health tool":
        "get_portfolio_health" in globals(),

    "Top risks tool":
        "get_top_risks" in globals(),

    "AI PM Copilot":
        "start_novabridge_chat" in globals(),

    "Human approval proposal":
        "propose_escalation" in globals(),

    "Human approval execution":
        "handle_escalation_approval" in globals(),

    "n8n escalation integration":
        "send_immediate_risk_to_n8n" in globals()
}

all_ready = True

for component, ready in components.items():

    status = "✅ READY" if ready else "❌ MISSING"

    print(
        f"{component:<32} {status}"
    )

    if not ready:
        all_ready = False


print("\n" + "=" * 60)

if all_ready:
    print("🚀 NOVABRIDGE SYSTEM STATUS: PRODUCTION DEMO READY")
else:
    print("⚠️ NOVABRIDGE SYSTEM STATUS: SOME COMPONENTS NEED ATTENTION")

print("=" * 60)

In [ ]:
# ============================================================
# PHASE 5 - FINAL DEMO
# NOVABRIDGE AI PM AGENT
# ============================================================

demo_question = """
Give me a concise executive update on the NovaBridge portfolio.
Tell me:
1. Overall portfolio health
2. Biggest current risk
3. The task causing that risk
4. What management should focus on next
"""

demo_answer = ask_novabridge_agent(demo_question)